# Station Stacking v18.1 RAP Wunderground Ablation - KDAL

Experimental notebook for `KDAL`.

This v18.1 RAP ablation keeps the v11 remaining-warmup ridge-stack backbone, requires Wunderground station-history labels, and adds only the coverage-gated RAP physics and station-specific physics shard features. It writes isolated artifacts to `data/calibration/station_stacking_v18_1_rap`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 100
OPTUNA_STARTUP_TRIALS = 40
STACK_OPTUNA_STARTUP_TRIALS = 40
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v18_1_rap_physics_settlement_stack"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V11 Contract

`feature_version="v18_1_rap"` adds only the coverage-gated RAP physics and station-specific physics shard features while selecting by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
6,KDAL,gfs,1992,2021-01-01,2026-06-21
7,KDAL,hrrr,1998,2021-01-01,2026-06-21
8,KDAL,nbm,1997,2021-01-01,2026-06-21


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v18_1_rap",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide_plus",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v18_1_rap",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v18_1_rap/KDAL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-07-11 13:40:00,538] A new study created in RDB with name: KDAL_v18_1_rap_remaining_warmup_base_xgboost_mae_f_wide_plus
[I 2026-07-11 13:40:34,399] Trial 0 finished with value: 1.5394757828127166 and parameters: {'n_estimators': 2278, 'learning_rate': 0.18404304074388048, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.5394757828127166.
[I 2026-07-11 13:41:24,719] Trial 1 finished with value: 1.5185793864170274 and parameters: {'n_estimators': 4263, 'learning_rate': 0.0005682336350005583, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 1 with value: 1.5185793864170274.
[I 2026-07-11 1

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,728,1.203560,1.667702
1,validation_2024_2025,lightgbm,728,1.210402,1.669266
2,validation_2024_2025,catboost,728,1.215077,1.671595
3,validation_2024_2025,provider_mean,728,2.372415,3.150214
4,validation_2024_2025,provider_median,728,2.034498,2.889602
5,validation_2024_2025,nbm_raw,728,1.882840,2.761133
6,validation_2024_2025,hrrr_raw,728,4.542543,5.430395
7,validation_2024_2025,gfs_raw,728,2.644711,3.533007
8,test_2026,xgboost,170,1.364361,1.783151
9,test_2026,lightgbm,170,1.367423,1.789349


In [8]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    target_source=config.effective_target_source,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/experiments/station_stacking_v18_1",
)

exported_weights.bundle_path, exported_weights.manifest_path


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v18_1_rap/model_weights/KDAL_station_high_regressor_v18_1_rap_physics_settlement_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v18_1_rap/model_weights/KDAL_station_high_regressor_v18_1_rap_physics_settlement_stack.json'))

## V11 Feature Coverage


In [9]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v2_spread_per_warmup_f,100.000000
1,v2_morning_warmup_to_consensus_f,100.000000
2,v3_remaining_warmup_from_high_so_far_f,100.000000
3,v3_high_so_far_above_current_f,100.000000
4,v2_humidity_warmup_interaction,100.000000
5,v4_forecast_wet_observed_dry,100.000000
6,v4_forecast_observed_precip_match,100.000000
7,v4_all_forecast_precip,100.000000
8,v3_humidity_remaining_warmup_interaction,100.000000
9,v3_remaining_warmup_per_spread_f,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
3,observed_temp_change_last_1h_f,numeric
4,observed_temp_change_last_3h_f,numeric
5,observed_morning_warmup_rate_f_per_hour,numeric
6,observed_high_so_far_change_since_9am_f,numeric
36,v2_recent_heat_anomaly_f,numeric
37,v2_recent_heat_momentum_f,numeric
38,v2_morning_warmup_to_consensus_f,numeric
39,v2_consensus_minus_7d_actual_f,numeric
40,v2_spread_per_warmup_f,numeric
41,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,100.0
1,observed_temp_change_last_3h_f,100.0
2,observed_morning_warmup_rate_f_per_hour,100.0
3,observed_high_so_far_change_since_9am_f,100.0


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
11,oof_2026,xgboost,170,112,65.882353
6,oof_2026,lightgbm,170,111,65.294118
10,oof_2026,ridge_stack,170,107,62.941176
4,oof_2026,guarded_blend_cap_3f,170,101,59.411765
0,oof_2026,catboost,170,99,58.235294
3,oof_2026,guarded_blend_cap_2f,170,96,56.470588
7,oof_2026,nbm_raw,170,84,49.411765
2,oof_2026,guarded_blend_cap_1f,170,79,46.470588
1,oof_2026,gfs_raw,170,69,40.588235
9,oof_2026,provider_median,170,69,40.588235


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,ridge_stack,138,1.369446,1.890651,v9
1,test_2026,xgboost,138,1.402066,1.889472,v11
2,test_2026,xgboost,138,1.419940,1.895994,v9
3,test_2026,ridge_stack,138,1.429989,1.907881,v11
4,test_2026,lightgbm,138,1.454606,1.932482,v11
...,...,...,...,...,...,...
83,validation_2024_2025,hrrr_raw,660,5.477361,6.266007,v3
84,validation_2024_2025,hrrr_raw,668,5.486824,6.274688,v7
85,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v5
86,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v6


## 2026 OOF Weather Brackets


In [15]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,170,1.364361,1.783151,1.345153,43.529412,3.603137,0.588235
1,lightgbm,170,1.367423,1.789349,1.344123,43.529412,3.682142,0.588235
2,catboost,170,1.481525,1.924130,1.379199,38.823529,3.738630,1.176471
3,ridge_stack,170,1.409834,1.833184,1.346043,44.117647,3.718634,0.588235
4,guarded_blend_cap_1f,170,2.031455,2.753680,1.602776,30.0,4.621835,4.117647
5,guarded_blend_cap_2f,170,1.730518,2.420628,1.537316,40.588235,4.204752,2.941176
6,guarded_blend_cap_3f,170,1.627322,2.266233,1.506963,40.588235,4.096856,2.352941
7,provider_mean,170,2.583543,3.303107,1.678281,23.529412,5.481044,8.235294
8,provider_median,170,2.428105,3.214617,1.757808,22.941176,5.950489,8.823529
9,nbm_raw,170,2.045105,2.795566,1.726940,34.117647,5.718984,8.235294
